# Failure analyst

One conversation over every flagged memory in NAMS, oldest first, numbered from 0. The model menu loads one of five analysts. Play stays on Gemma. Neo4j has to be up.

- Inkling 2 Mini (`thinkingmachines/Inkling-2-Mini`), local, unfinetuned
- Qwen3.8 27B (`Qwen/Qwen3.8-27B`), local
- Gemma 4 31B (`google/gemma-4-31B-it`), local
- GPT-6 Astra (`gpt-6-astra`), `OPENAI_API_KEY` in `.env`
- Claude Fable 5.1 (`claude-fable-5-1`), `ANTHROPIC_API_KEY` in `.env`

**Load model** loads the selected row. Changing the menu does not download anything. The first load of a local model pulls its weights.

The reply can end with one tool token. The notebook shows the result and the model continues, up to 24 tools, then the turn comes back.

- `[LIST]` — one line per flag: number, game, flag text
- `[LOAD n]` — both questions, both messages, the before picture and its Settings
- `[LOAD n: parts]` — only those parts. `bq bm bi bs aq am ai as gaze`. `b`/`a` is before/after; `q` question, `m` message, `i` image, `s` settings
- `[DROP n]`, `[DROP n: parts]`, `[DROP ALL]`
- `[GAZE n]` — s and v on the before message: at the start, at the end of each line, and at the final `]`
- `[GAZE n: a-b]` — every token from a to b. `[GAZE n: -k]` is the last k tokens
- `[NOTE key: text]` and `[FORGET key]` — the scratchpad, not a tip
- `[SEARCH words]` — flagged messages whose text matches

At most 4 pictures stay loaded. A loaded memory stays until it is dropped. **Restart thread** starts a new analyst conversation and clears the loaded set. **Reset to pre-analyst** deletes every analyst conversation, message, note, and reasoning trace, then does that restart. Play flags and snapshots stay, and the loaded model stays. Floats on screen are three decimals. Stored traces are not rewritten.



In [ ]:
import html
import os
import sys
import traceback
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets

from agent.analyst import FailureAnalyst
from agent.inkling import ANALYST_MODELS
from agent.notebook_ui import tame_shift_enter

state = {"analyst": None, "busy": False}


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag


def _selected_label():
    key = model_menu.value
    for spec in ANALYST_MODELS.values():
        if spec.key == key:
            return spec.label
    raise RuntimeError("unknown analyst model %r" % (key,))


def _refresh_side():
    analyst = state["analyst"]
    if analyst is None:
        loaded.value = "<pre>Nothing loaded.</pre>"
        pad.value = "<pre>No thread.</pre>"
        thumbs.children = ()
        return
    loaded.value = "<pre>" + html.escape(analyst.catalog()) + "</pre>"
    pad.value = "<pre>" + analyst.notepad() + "</pre>"
    children = []
    for item in analyst.loaded_images():
        file = Path(item["path"])
        if not file.is_file():
            raise FileNotFoundError("loaded image is missing: %s" % (file,))
        children.append(widgets.VBox([
            widgets.HTML(
                "<b>memory %s %s</b>" % (item["n"], item["side"])
            ),
            widgets.Image(
                value=file.read_bytes(), format="png", width=240,
            ),
        ]))
    thumbs.children = tuple(children)


def on_load(_):
    if state["busy"]:
        return
    _set_busy(True)
    label = _selected_label()
    status.value = "Loading " + label + "."
    try:
        if state["analyst"] is None:
            state["analyst"] = FailureAnalyst(load_model=False)
        info = state["analyst"].load_analyst_model(model_menu.value)
        status.value = "Loaded " + str(info["label"]) + "."
        _say(info["model_id"])
        _refresh_side()
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_ask(_):
    if state["busy"]:
        return
    analyst = state["analyst"]
    if analyst is None or analyst.model is None:
        status.value = "Load a model first."
        return
    question = box.value.strip()
    if not question:
        status.value = "Write a question."
        return
    _set_busy(True)
    status.value = "Asking."
    chunks = []

    def _step(step):
        if step.get("text"):
            chunks.append(step["text"])
            reply.value = "<pre>" + "\n\n".join(chunks) + "</pre>"
        if step.get("tool_result") is not None:
            chunks.append(step["tool_result"])
            reply.value = "<pre>" + "\n\n".join(chunks) + "</pre>"
            _refresh_side()

    try:
        analyst.ask(question, on_step=_step)
        status.value = "Done."
        _refresh_side()
    except Exception:
        status.value = "Ask failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_restart(_):
    if state["busy"]:
        return
    analyst = state["analyst"]
    if analyst is None:
        status.value = "No thread yet."
        return
    info = analyst.restart()
    reply.value = "<pre></pre>"
    status.value = "New thread."
    _say("thread " + info["session_id"])
    _refresh_side()


def on_reset(_):
    if state["busy"]:
        return
    _set_busy(True)
    status.value = "Resetting to pre-analyst."
    try:
        if state["analyst"] is None:
            state["analyst"] = FailureAnalyst(load_model=False)
        info = state["analyst"].reset_to_pre_analyst()
        reply.value = "<pre></pre>"
        status.value = "Pre-analyst. New thread."
        _say("thread " + info["session_id"])
        _refresh_side()
    except Exception:
        status.value = "Reset failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


model_menu = widgets.Dropdown(
    options=[(spec.label, spec.key) for spec in ANALYST_MODELS.values()],
    value="inkling-2-mini",
    description="Model",
    layout=widgets.Layout(width="320px"),
)
load_btn = widgets.Button(description="Load model", button_style="primary")
box = widgets.Textarea(
    value="What do these flags have in common?",
    layout=widgets.Layout(width="100%", height="80px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart thread")
reset_btn = widgets.Button(
    description="Reset to pre-analyst",
    layout=widgets.Layout(width="200px"),
)
status = widgets.HTML("<span>Pick a model and load it.</span>")
reply = widgets.HTML("<pre></pre>")
loaded = widgets.HTML("<pre>Nothing loaded.</pre>")
pad = widgets.HTML("<pre>No thread.</pre>")
thumbs = widgets.HBox()
log = widgets.Output(layout=widgets.Layout(height="120px", overflow="auto"))

controls = [model_menu, load_btn, box, ask_btn, restart_btn, reset_btn]
load_btn.on_click(on_load)
ask_btn.on_click(on_ask)
restart_btn.on_click(on_restart)
reset_btn.on_click(on_reset)

tame_shift_enter(box)
state["analyst"] = FailureAnalyst(load_model=False)
_refresh_side()
widgets.VBox([
    widgets.HBox([model_menu, load_btn]),
    status,
    box,
    widgets.HBox([ask_btn, restart_btn, reset_btn]),
    widgets.HTML("<b>Reply</b>"),
    reply,
    widgets.HTML("<b>Loaded</b>"),
    loaded,
    widgets.HTML("<b>Scratchpad</b>"),
    pad,
    widgets.HTML("<b>Loaded images</b>"),
    thumbs,
    log,
])
